# Mixed-effects models: AD versus HC in Cluster 2

Marianna Georgiou, Department of Psychiatry, Psychotherapy and Psychosomatics, Faculty of Medicine, RWTH Aachen, Aachen, Germany
**Email:** mariannag43@gmail.com
**Publication:**  Latent subtypes of longitudinal postpartum trajectories for identifying healthy women at risk for affective conditions

Compare linear score trajectories between AD and HC participants in Cluster 2, separately in discovery and validation cohorts. Fit one random-intercept model per measure and export coefficient summaries.

## Inputs

All inputs are in `files/interm_files/`:

- `kmeans[1, 2].xlsx` and `kmeans[3].xlsx`: normalized clustering features with ID, CODE, and IDX coded 0/1.
- `disc_merged_rawdata_withlabels.xlsx` and `val_merged_rawdata_withlabels.xlsx`: raw MPAS T1–T4 with HC/AD/PPD labels and cluster labels 1/2.

The code converts the clustering labels to 1/2 and ND/AS to HC/AD before joining raw MPAS on ID/CODE/IDX. IDs are assumed unique. Five outcomes use the normalized clustering features; MPAS remains on its raw scale.

## Output

`files/interm_files/mixed_effects_model_supplementary.xlsx`: coefficient summaries for six measures in each cohort, including cohort, measure, and available participant count. No row index is exported. Stored notebook outputs are cleared.


## 1. Import dependencies

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import chi2


## 2. Load features and attach raw MPAS

Join MPAS by participant, diagnosis, and displayed cluster label using a left join. Unmatched MPAS values remain missing.

In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd()
OUTPUT_FILES = BASE_DIR / "files" / "interm_files"
OUTPUT_FILES.mkdir(parents=True, exist_ok=True)

disc_path = OUTPUT_FILES / 'kmeans[1, 2].xlsx'
disc = pd.read_excel(disc_path)
disc["CODE"] = disc["CODE"].replace({
    "ND": "HC",
    "AS": "AD"
})
disc["IDX"] = disc["IDX"].replace({
    0: 1,
    1: 2
})

path_for_MPAS_disc = OUTPUT_FILES / "disc_merged_rawdata_withlabels.xlsx"
for_MPAS_disc = pd.read_excel(path_for_MPAS_disc)
mpas_disc = for_MPAS_disc[['MPAS T1', 'MPAS T2', 'MPAS T3','MPAS T4', 'CODE', 'IDX', 'ID']]
disc = disc.merge(
    mpas_disc,
    on=["CODE", "IDX", 'ID'],
    how="left"
)

val_path = OUTPUT_FILES / 'kmeans[3].xlsx'
val = pd.read_excel(val_path)
val["CODE"] = val["CODE"].replace({
    "ND": "HC",
    "AS": "AD"
})
val["IDX"] = val["IDX"].replace({
    0: 1,
    1: 2
})

path_for_MPAS_val = OUTPUT_FILES / "val_merged_rawdata_withlabels.xlsx"
for_MPAS_val = pd.read_excel(path_for_MPAS_val)
mpas_val = for_MPAS_val[['MPAS T1', 'MPAS T2', 'MPAS T3','MPAS T4', 'CODE', 'IDX', 'ID']]
val = val.merge(
    mpas_val,
    on=["CODE", "IDX", "ID"],
    how="left"
)


In [ ]:
disc


In [ ]:
disc["CODE"].value_counts()


## 3. Select HC and AD in Cluster 2

Both cohorts are filtered to IDX=2 after relabeling. Print diagnosis counts for the selected participants.

In [ ]:
disc_HC_AD = disc[
    (disc["IDX"] == 2) &
    (disc["CODE"].isin(["HC", "AD"]))
].copy()

val_HC_AD = val[
    (val["IDX"] == 2) &
    (val["CODE"].isin(["HC", "AD"]))
].copy()

print("Discovery:")
print(disc_HC_AD["CODE"].value_counts())

print("\nValidation:")
print(val_HC_AD["CODE"].value_counts())


## 4. Fit and export mixed-effects models

Reshape each outcome to long format and drop missing scores. Time is numeric 0–4 (1–4 for MPAS), treating successive time labels as equally spaced. Fit `Score ~ CODE * Time` with HC as reference and a participant-specific random intercept; no random slope is specified. `.fit()` retains statsmodels defaults, including REML.

The interaction estimates the AD–HC difference in linear slope. The intercept and group coefficient refer to Time=0; for MPAS this is extrapolated because observations start at T1. The exported “HC at T0” label is retained. Models are fitted separately for six outcomes in each cohort without a multiple-testing correction. Inspect printed fit warnings and summaries for convergence before interpreting results.

In [ ]:


measures = {
    "Anhedonia":  [f"Anhedonie_t{i}" for i in range(5)],
    "Depression": [f"Depression_t{i}" for i in range(5)],
    "Anxiety":    [f"Anxiety_t{i}" for i in range(5)],
    "Mood":       [f"Mood T{i}" for i in range(5)],
    "Stress":     [f"Stress T{i}" for i in range(5)],
    "MPAS":       [f"MPAS T{i}" for i in range(1, 5)]
}

all_model_tables = []

def run_mixed_models(df, cohort_name):
    """Fit each outcome and append its coefficient summary to all_model_tables."""

    for measure, columns in measures.items():

        long_df = df.melt(
            id_vars=["ID", "CODE"],
            value_vars=columns,
            var_name="Timepoint",
            value_name="Score"
        ).dropna(subset=["Score"])

        long_df["Time"] = (
            long_df["Timepoint"]
            .str.extract(r"(\d)$")[0]
            .astype(int)
        )

        long_df["CODE"] = pd.Categorical(
            long_df["CODE"],
            categories=["HC", "AD"]
        )

        # Group-by-linear-time fixed effects with a participant random intercept.
        model = smf.mixedlm(
            "Score ~ CODE * Time",
            long_df,
            groups=long_df["ID"]
        ).fit()

        print("\n" + "=" * 80)
        print(f"{cohort_name} — {measure}")
        print("=" * 80)

        print(f"N participants = {long_df['ID'].nunique()}")

        print(model.summary())

        table = model.summary().tables[1].copy()

        table = table.rename(index={
            "Intercept": "Intercept (HC at T0)",
            "CODE[T.AD]": "Group (AD vs. HC)",
            "Time": "Time",
            "CODE[T.AD]:Time": "Group (AD vs. HC) × Time",
            "Group Var": "Random intercept variance (ID)"
        })

        table = table.reset_index()

        table = table.rename(
            columns={"index": "Model term"}
        )

        table.insert(0, "Cohort", cohort_name)
        table.insert(1, "Measure", measure)
        table.insert(
            2,
            "N participants",
            long_df["ID"].nunique()
        )

        all_model_tables.append(table)

run_mixed_models(
    disc_HC_AD,
    "Discovery"
)

run_mixed_models(
    val_HC_AD,
    "Validation"
)

final_table = pd.concat(
    all_model_tables,
    ignore_index=True
)

output_path = (
    OUTPUT_FILES / "mixed_effects_model_supplementary.xlsx"
)

final_table.to_excel(
    output_path,
    index=False
)

print("\n" + "=" * 80)
print("EXCEL FILE SAVED SUCCESSFULLY")
print("=" * 80)
print(output_path)
